<a href="https://colab.research.google.com/github/jaloaiza/genaiassignments/blob/main/assignments/06/Module6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generate Synthetic Training Data

<a target="_blank" href="https://colab.research.google.com/github/simonguest/CS-394/blob/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://github.com/simonguest/CS-394/raw/refs/heads/main/src/06/notebooks/generate-synthetic.ipynb">
  <img src="https://img.shields.io/badge/Download_.ipynb-blue" alt="Download .ipynb"/>
</a>

## Data generation settings

In [60]:
NUM_TRAIN_EXAMPLES = 10  # @param {type:"number"}
NUM_VAL_EXAMPLES = 3  # @param {type:"number"}
NUM_TEST_EXAMPLES = 3 # @param {type:"number"}
TEMPERATURE = 0.8  # @param {type:"number"}

DATA_FOLDER = "./data/generated"
!mkdir -p {DATA_FOLDER}

DATAGEN_URL = "https://openrouter.ai/api/v1"
DATAGEN_MODEL = "openai/gpt-5-nano"

# Create it if it doesn't exist
os.makedirs(DATA_FOLDER, exist_ok=True)

# Check if it exists
print("Folder exists?", os.path.exists(DATA_FOLDER))
print("Full path:", os.path.abspath(DATA_FOLDER))

Folder exists? True
Full path: /content/data/generated


## Dataset diversity

In [61]:
TOPICS= [
    "animals_common", # START CHILD
    "colors_shapes",
    "weather_basic",
    "toys",
    "school_objects",
    "family_roles",
    "food_simple",
    "technology_basic", # START TEEN
    "friendship",
    "emotions_basic",
    "sports_common",
    "school_life",
    "music_general",
    "time_concepts",
    "dreams_goals",
    "work_life", # START ADULT
    "money",
    "relationships",
    "parenthood",
    "memory",
    "aging",
    "transportation",
    "communication",
    "habits"
]

DIFFICULTY_LEVELS = [
    "easy",
    "medium",
    "hard",
    "trick"
]

DIFFICULTY_LEVEL_WEIGHTS = [0.4, 0.3, 0.2, 0.1]

RIDDLE_LENGTH = [
    "short",
    "medium",
    "paragraph"
]

RIDDLE_LENGTH_WEIGHTS = [0.4, 0.35, 0.25]


## Model for structured output

In [62]:
from pydantic import BaseModel

class RiddleExample (BaseModel):
    topic: str
    difficulty: str
    riddle_length: str
    riddle: str
    answer: str
    explanation: str

## Get OpenRouter API key

In [63]:
import sys
import os
from dotenv import load_dotenv

if 'google.colab' in sys.modules:
  from google.colab import userdata # type:ignore
  os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
else:
  load_dotenv()

## Conversation generation functions

In [64]:
from httpx._transports.default import A
import openai
import os
from dataclasses import dataclass

client = openai.OpenAI(
    base_url=DATAGEN_URL,
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

def generate_completion(prompt: str) -> RiddleExample | None:
    response = client.responses.parse(
        model=DATAGEN_MODEL,
        input=[{"role": "user", "content": prompt}],
        temperature=TEMPERATURE,
        stream=False,
        text_format=RiddleExample
    )

    return response.output_parsed

def map_riddle_length(riddle_length: str, topic: str) -> str:
    if riddle_length == "short":
        return f"a short riddle (1–2 sentences) about {topic}"
    elif riddle_length == "medium":
        return f"a medium-style riddle (3-4 lines) about {topic}"
    elif riddle_length == "paragraph":
        return f"a short story-style riddle (5–6 lines) about {topic}"
    else:
        return f"a riddle about {topic}"

def create_conversation(
    topic: str,
    difficulty_level: str,
    riddle_length: str) -> RiddleExample | None:

    request = map_riddle_length(riddle_length, topic)

    prompt = f"""
    Generate {request}.

    Constraints:
    - Difficulty level: {difficulty_level}
    - The riddle should be of length: {riddle_length}
    - The riddle should be about {topic}

    Answer requirements:
    - The answer should be 1-3 words long.
    - The answer must logically match the riddle.
    - Do not reveal the answer inside the riddle.

    Return in the following format with the following:

    RIDDLE:
    <riddle text>

    ANSWER:
    <answer only>

    EXPLANATION:
    <Correct/Incorrect>
    """

    return generate_completion(prompt)

## Dataset generation functions

In [65]:
import random
import json
from tqdm import tqdm

def generate_dataset(num_examples: int, filename: str) -> None:
    with open(filename, "w", encoding="utf-8") as f:
        for idx in tqdm(range(num_examples)):

          topic = random.choice(TOPICS)

          difficulty_level = random.choices(
              DIFFICULTY_LEVELS,
              weights=DIFFICULTY_LEVEL_WEIGHTS
          )[0]

          riddle_length = random.choices(
              RIDDLE_LENGTH,
              weights=RIDDLE_LENGTH_WEIGHTS
          )[0]

          # Generate riddle
          conversation = None
          while conversation is None:
              conversation = create_conversation(
                  topic=topic,
                  difficulty_level = difficulty_level,
                  riddle_length = riddle_length
              )

          # Build the user prompt
          user_prompt = (
              f"Create a {difficulty_level} {riddle_length} riddle "
              f"about {topic} suitable for a family friendly audience. "
              f"The answer should be a 1-3 words."
          )

          # Structured training template
          template = {
          "messages": [
              {
                  "role": "user",
                  "content": user_prompt
              },
              {
                  "role": "assistant",
                  "content": f"""RIDDLE:
                  {conversation.riddle}

                  ANSWER:
                  {conversation.answer}

                  EXPLANATION:
                  {conversation.explanation}
                  """
              }
              ]
          }

          line = json.dumps(template, ensure_ascii=False) + "\n"
          f.write(line)
          f.flush()

        f.flush()
        f.close()

## Generate all the data!

In [66]:
from datetime import datetime

TRAIN_FILE = f"{DATA_FOLDER}/train_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
VALID_FILE = f"{DATA_FOLDER}/valid_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"
TEST_FILE = f"{DATA_FOLDER}/test_{datetime.now().strftime('%Y-%m-%d-%H:%M:%S')}.jsonl"

generate_dataset(NUM_TRAIN_EXAMPLES, TEST_FILE)


100%|██████████| 10/10 [02:57<00:00, 17.72s/it]
